# Epoch별 Shortcut Learning 실험 — Colab 실행 가이드

이 노트북은 제공된 ZIP 파일의 `shortcut_experiment.py`, `analyze.py`를 사용하는 실행용 가이드입니다. **GPU 런타임 권장**. 데이터셋은 Hugging Face Hub에서 내려받습니다.

실험은 `(VGG-11-BN, ResNet-18) × (Colored MNIST, Waterbirds)`의 4가지 조합을 지원하고, 매 epoch 모델·optimizer·scheduler 체크포인트 및 hidden-feature `.npz`를 저장합니다.

## 1. ZIP 업로드 및 설치

같이 제공된 `shortcut_learning_experiment.zip` 파일을 업로드하세요.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith(".zip")), None)
assert archive is not None, "shortcut_learning_experiment.zip 을 업로드하세요"
with zipfile.ZipFile(archive) as f:
    f.extractall("/content")
os.chdir("/content/shortcut_learning_experiment")
print("Working directory:", os.getcwd())

In [ ]:
%pip -q install -r requirements.txt

## 2. 첫 번째 실험 — Colored MNIST + ResNet-18

`train-corr=0.95`, `val-corr=0.50`, `test-corr=0.10`입니다. 기본 결과는 `/content/shortcut_learning_experiment/runs/cm_resnet`에 저장됩니다. 매 epoch 검증 데이터의 WGA, 그룹별 정확도, 색상 flip rate를 기록합니다.

In [ ]:
!python shortcut_experiment.py train --dataset colored_mnist --model resnet18 --epochs 10 --batch-size 64 --num-workers 2 --output runs/cm_resnet

## 3. Epoch별 결과 및 중간 feature 확인

In [ ]:
!python analyze.py --run runs/cm_resnet --epoch 5 --layer layer3
from IPython.display import display, Image as IPImage
display(IPImage(filename="runs/cm_resnet/learning_curves.png"))
display(IPImage(filename="runs/cm_resnet/hidden_pca_epoch_005.png"))

In [ ]:
import numpy as np
with np.load("runs/cm_resnet/features/epoch_005.npz") as z:
    print("Recorded layers:", z["layers"])
    print("Targets:", z["labels"].shape, "Shortcut labels:", z["spurious"].shape)
    for i, name in enumerate(z["layers"]):
        print(name, z[f"f{i}"].shape)  # [N, feature_dim]

### 원본 중간 feature map (평균 풀링 전)

단일 샘플의 `layer3` activation `[C,H,W]`을 추출하여 `.npz`로 저장하고 여러 채널을 이미지로 시각화합니다.

In [ ]:
!python inspect_hidden.py --run runs/cm_resnet --checkpoint runs/cm_resnet/checkpoints/epoch_005.pt --split val --sample-index 0 --layer layer3 --channels 16
from pathlib import Path
png = next(Path("runs/cm_resnet").glob("raw_epoch_005_val_0_layer3.png"))
display(IPImage(filename=str(png)))

## 4. 체크포인트 직접 복원 / 평가

체크포인트에는 단순 모델 가중치뿐 아니라 optimizer, scheduler, RNG 상태가 들어 있습니다.

In [ ]:
import torch
from shortcut_experiment import build_model
ckpt_path = "runs/cm_resnet/checkpoints/epoch_005.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
restored = build_model(ckpt["config"])
restored.load_state_dict(ckpt["model"])
restored.eval()
print("Restored epoch:", ckpt["epoch"])
print("Optimizer state available:", "optimizer" in ckpt)

In [ ]:
!python shortcut_experiment.py eval --run runs/cm_resnet --checkpoint runs/cm_resnet/checkpoints/epoch_005.pt --split test

## 5. 나머지 세 실험

VGG는 계산/메모리 사용량이 더 큽니다. Waterbirds는 ImageNet 사전학습을 사용하는 편이 학습 데이터 규모상 실용적입니다. 아래 셀에서 필요한 명령만 실행하세요.

In [ ]:
# Colored MNIST + VGG-11-BN
# !python shortcut_experiment.py train --dataset colored_mnist --model vgg11_bn --epochs 20 --output runs/cm_vgg

# Waterbirds + ResNet-18
# !python shortcut_experiment.py train --dataset waterbirds --model resnet18 --pretrained --epochs 20 --batch-size 32 --output runs/wb_resnet

# Waterbirds + VGG-11-BN
# !python shortcut_experiment.py train --dataset waterbirds --model vgg11_bn --pretrained --epochs 20 --batch-size 32 --output runs/wb_vgg

## 6. 중단 후 학습 재개 / 결과 백업

Colab 임시 저장소는 세션이 종료되면 사라질 수 있으므로 `runs/` 디렉터리를 Drive에 복사하거나 별도로 다운로드하는 것을 권장합니다.

In [ ]:
# 같은 run 디렉터리에서 가장 최근 epoch부터 재개
# !python shortcut_experiment.py train --dataset colored_mnist --model resnet18 --output runs/cm_resnet --resume latest

# 결과 전체를 ZIP으로 압축 (체크포인트가 크므로 충분한 저장 공간 필요)
# !zip -qr runs_export.zip runs/
# files.download("runs_export.zip")